In [15]:
# 1. Install deps

!pip install pandas scikit-learn gensim -q


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import pandas as pd
import os
import sys

In [17]:
if 'google.colab' in sys.modules: 
    if not os.path.exists('/content/nlp_uni'):
        !git clone -b lab-08 https://github.com/Danylo-NULP/nlp_uni.git
    
    %cd /content/nlp_uni
    !pip install pandas scikit-learn spacy -q
    sys.path.append('/content/nlp_uni')
    
    FOLDER_ID = '1LhS2rA8VAQVd_lzUwMXuHav6fSVcGO0D'
    
    os.makedirs('/content/nlp_uni/data', exist_ok=True)
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O /content/nlp_uni/data/
    
    data_dir = '/content/nlp_uni/data'

else:
    sys.path.append(os.path.abspath('..'))
    data_dir = '../data'

In [18]:
# 3. Corpus filtering / preprocessing checks

sys.path.append(os.path.abspath('..') if not 'google.colab' in sys.modules else '/content/nlp_uni')
from src.topic_modeling import build_lsa_pipeline, build_lda_pipeline
from src.topic_utils import get_top_words, get_top_documents

# Завантажуємо датасет (використовуємо premise_clean, бо це опис оригінального зображення)
data_path = f'{data_dir}/processed_v2/processed_v2.csv'
df = pd.read_csv(data_path)

len_before = len(df)

# Прибираємо порожні значення
df_filtered = df.dropna(subset=['premise_clean']).copy()

# Для Topic Modeling нам потрібні довші тексти, де є хоч якийсь контекст
df_filtered = df_filtered[df_filtered['premise_clean'].str.len() > 15]

len_after = len(df_filtered)

print(f"Кількість документів до фільтрації: {len_before}")
print(f"Кількість документів після фільтрації: {len_after}")
print(f"Видалено занадто коротких/порожніх: {len_before - len_after}")

corpus = df_filtered['premise_clean'].astype(str)

Кількість документів до фільтрації: 1500
Кількість документів після фільтрації: 1499
Видалено занадто коротких/порожніх: 1


In [19]:
# 4. LSA experiments

print("Запуск експериментів LSA (Latent Semantic Analysis)")

# Для LSA (TF-IDF) корисно використовувати і біграми, щоб ловити фрази типу "little boy"
lsa_pipe_5 = build_lsa_pipeline(n_topics=5, ngram_range=(1, 2), min_df=5, max_df=0.8)
lsa_pipe_5.fit(corpus)

lsa_pipe_8 = build_lsa_pipeline(n_topics=8, ngram_range=(1, 2), min_df=5, max_df=0.8)
lsa_pipe_8.fit(corpus)

lsa_pipe_10 = build_lsa_pipeline(n_topics=10, ngram_range=(1, 2), min_df=5, max_df=0.8)
lsa_pipe_10.fit(corpus)

print("LSA моделі успішно навчені")

Запуск експериментів LSA (Latent Semantic Analysis)
LSA моделі успішно навчені


In [20]:
# 5. LDA experiments

print("Запуск експериментів LDA (Latent Dirichlet Allocation)")

# Для LDA (CountVectorizer) зазвичай беруть лише уніграми, щоб не розмивати ймовірності
lda_pipe_5 = build_lda_pipeline(n_topics=5, ngram_range=(1, 1), min_df=5, max_df=0.8)
lda_pipe_5.fit(corpus)

lda_pipe_8 = build_lda_pipeline(n_topics=8, ngram_range=(1, 1), min_df=5, max_df=0.8)
lda_pipe_8.fit(corpus)

lda_pipe_10 = build_lda_pipeline(n_topics=10, ngram_range=(1, 1), min_df=5, max_df=0.8)
lda_pipe_10.fit(corpus)

print("LDA моделі успішно навчені")

Запуск експериментів LDA (Latent Dirichlet Allocation)
LDA моделі успішно навчені


In [21]:
# 6. Top words per topic

def display_topics(pipeline, name, k):
    print(f"\n--- {name} (k={k}) Топ-10 слів ---")
    topics = get_top_words(pipeline, n_words=10)
    for topic_idx, words in topics.items():
        print(f"Topic {topic_idx}: {', '.join(words)}")

display_topics(lsa_pipe_5, "LSA", 5)
display_topics(lsa_pipe_10, "LSA", 10)

display_topics(lda_pipe_5, "LDA", 5)
display_topics(lda_pipe_10, "LDA", 10)


--- LSA (k=5) Топ-10 слів ---
Topic 0: man, woman, shirt, wearing, blue, white, black, people, street, young
Topic 1: people, group, street, group people, walking, men, people walking, women, building, standing
Topic 2: man, people, street, man standing, man playing, man sitting, walking, man walking, man black, people walking
Topic 3: woman, street, man woman, woman sitting, looking, cellphone, talking, woman wearing, woman looking, walking street
Topic 4: men, women, walking, street, sitting, working, bench, stand, sidewalk, men sitting

--- LSA (k=10) Топ-10 слів ---
Topic 0: man, woman, shirt, wearing, blue, white, black, people, street, young
Topic 1: people, group, street, group people, walking, men, people walking, women, building, standing
Topic 2: man, people, street, man standing, man playing, man sitting, riding, man walking, man wearing, man black
Topic 3: woman, street, man woman, woman sitting, cellphone, looking, talking, woman wearing, woman looking, walking
Topic 4: m

In [22]:
# 7. Top documents per topic

def display_top_docs(pipeline, corpus, name, k, n_docs=2):
    print(f"\n=== ТОП-ДОКУМЕНТИ ДЛЯ: {name} (k={k}) ===")
    top_docs = get_top_documents(pipeline, corpus, n_docs=n_docs)
    for topic_idx, docs in top_docs.items():
        print(f"\nTopic {topic_idx}:")
        for i, (doc, score) in enumerate(docs):
            print(f"  [Score: {score:.3f}] {doc}")

display_top_docs(lsa_pipe_10, corpus, "LSA", 10)
display_top_docs(lda_pipe_10, corpus, "LDA", 10)


=== ТОП-ДОКУМЕНТИ ДЛЯ: LSA (k=10) ===

Topic 0:
  [Score: 0.569] A woman is checking a man's blood pressure at home.
  [Score: 0.529] This man is repairing the pipes.

Topic 1:
  [Score: 0.710] A whole bunch of people in church.
  [Score: 0.710] People toss candy as they celebrate on top of a float.

Topic 2:
  [Score: 0.679] A man is praying to an idol.
  [Score: 0.679] man washing the windows of a skyscraper

Topic 3:
  [Score: 0.758] A woman grilling chicken and veggies.
  [Score: 0.758] A woman with a designer handbag is admiring the view.

Topic 4:
  [Score: 0.850] Two men in a laboratory are peering through microscopes to observe slides.
  [Score: 0.850] Two men skateboarding in the forest.

Topic 5:
  [Score: 0.468] Young boy turning a crank in the street.
  [Score: 0.434] A young girl wears a butterfly costume.

Topic 6:
  [Score: 0.688] Two women are walking casually down the street together.
  [Score: 0.619] A street performer's drum beats gain many onlookers.

Topic 7:
  [S

In [23]:
# 8. Manual interpretation

print("""
[РУЧНА ІНТЕРПРЕТАЦІЯ ТЕМ (для моделі LDA k=10)]

Вступ: 
Корпус SNLI складається з описів випадкових фотографій. Тому наші теми — це візуальні сюжети (спортивні ігри, вуличні сцени, одяг). Ми аналізуємо LDA (k=10), оскільки ця модель змогла виділити специфічні сценарії, на відміну від LSA, яка просто змішала загальні слова "man, woman, street".

* Topic 0: Ділові та робочі групи
  * Пояснення: Тема групує людей за роботою або діловими зустрічами. Слова "building", "working", "asian" підкріплюються документами про архітектурну модель та азійського бізнесмена.
* Topic 1: Жінки позують / Фотографування
  * Пояснення: Наявність слів "camera", "smiling", "girls", "posing". Документи описують жінок, які стоять разом, дивляться в камеру або позують (наприклад, з віолончеллю).
* Topic 2: Дозвілля різних поколінь
  * Пояснення: Слова "old", "young", "playing", "park". Документи підтверджують це: дідусь вчить онука плавати, дитина грає у відеогру.
* Topic 3: Дитяча активність та басейни
  * Пояснення: Слова "boy", "little", "jumping", "blue". У топ-документах яскраво виділяються хлопчики, що стрибають у басейн, та бейсболісти.
* Topic 4: Детальні описи одягу
  * Пояснення: Тема сфокусована суто на візуальному описі вбрання. Топ-слова: "wearing", "shirt", "hat", "shorts". Топ-документи детально описують гавайські спідниці, краватки та блузки.
* Topic 5: Командні види спорту (Футбол)
  * Пояснення: Найчистіша тема. Слова "soccer", "ball", "field", "running". Обидва топ-документи ідеально описують гру у футбол (гравці в синьому та помаранчевому).
* Topic 6: Чоловіки за ремеслом / з інструментами
  * Пояснення: Слова "stands", "green", "man". Документи описують чоловіка з мечем у лісі та коваля (blacksmith) за роботою біля вогню.
* Topic 7: Шопінг та ресторани
  * Пояснення: Слова "food", "eating", "restaurant", "holding". Документи описують чоловіка з пакетами біля візка (шопінг).
* Topic 8: Вуличні взаємодії та емоції
  * Пояснення: Слова "bench", "outside", "watching", "screaming". Документи ілюструють вуличну стрижку бороди та жінку, що кричить на тротуарі.
* Topic 9: Музичні гурти (Змішана)
  * Пояснення: Наявність слів "band", "guitars", "equipment" у документах чітко вказує на музикантів, хоча топ-слова теми дещо розмиті загальними "man", "woman".
""")


[РУЧНА ІНТЕРПРЕТАЦІЯ ТЕМ (для моделі LDA k=10)]

Вступ: 
Корпус SNLI складається з описів випадкових фотографій. Тому наші теми — це візуальні сюжети (спортивні ігри, вуличні сцени, одяг). Ми аналізуємо LDA (k=10), оскільки ця модель змогла виділити специфічні сценарії, на відміну від LSA, яка просто змішала загальні слова "man, woman, street".

* Topic 0: Ділові та робочі групи
  * Пояснення: Тема групує людей за роботою або діловими зустрічами. Слова "building", "working", "asian" підкріплюються документами про архітектурну модель та азійського бізнесмена.
* Topic 1: Жінки позують / Фотографування
  * Пояснення: Наявність слів "camera", "smiling", "girls", "posing". Документи описують жінок, які стоять разом, дивляться в камеру або позують (наприклад, з віолончеллю).
* Topic 2: Дозвілля різних поколінь
  * Пояснення: Слова "old", "young", "playing", "park". Документи підтверджують це: дідусь вчить онука плавати, дитина грає у відеогру.
* Topic 3: Дитяча активність та басейни
  * Поя

In [24]:
# 9. Coherence Score (LSA vs LDA)

from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel

# Готуємо токенізовані тексти для Gensim (розбиваємо по пробілах)
texts_tokenized = [text.lower().split() for text in corpus]
dictionary = Dictionary(texts_tokenized)

def calculate_coherence_score(pipeline, name, k):
    # Отримуємо списки слів з нашої функції
    topics_dict = get_top_words(pipeline, n_words=10)
    topics_list = [words for _, words in topics_dict.items()]
    
    # Рахуємо c_v coherence
    cm = CoherenceModel(topics=topics_list, texts=texts_tokenized, dictionary=dictionary, coherence='c_v')
    score = cm.get_coherence()
    print(f"Coherence Score для {name} (k={k}): {score:.4f}")
    return score

print("Оцінка Coherence")
calculate_coherence_score(lsa_pipe_5, "LSA", 5)
calculate_coherence_score(lsa_pipe_10, "LSA", 10)

calculate_coherence_score(lda_pipe_5, "LDA", 5)
calculate_coherence_score(lda_pipe_10, "LDA", 10)

Оцінка Coherence
Coherence Score для LSA (k=5): 0.3729
Coherence Score для LSA (k=10): 0.2982
Coherence Score для LDA (k=5): 0.3324
Coherence Score для LDA (k=10): 0.3456


np.float64(0.3455672197517549)

In [25]:
# 10. “Bad topics” analysis

print("""
[АНАЛІЗ "ПОГАНИХ" ТЕМ]

1. Проблемна тема: LSA (k=10), Теми 0, 1, 2, 3 та 4 (Duplicate Topics)
* Яка це тема: Майже половина тем у LSA є абсолютними дублікатами. Топ-слова у всіх цих п'яти темах складаються з міксу "man, woman, street, walking, sitting". 
* Чому вийшла поганою: Алгоритм SVD, на якому базується LSA, намагається знайти ортогональні вектори дисперсії. Оскільки в датасеті SNLI кожне речення починається з "A man..." або "A woman...", ці слова мають гігантську вагу в TF-IDF. LSA не зміг "пробитися" крізь цей лексичний шум і просто розмножив одну тему на п'ять.
* Що змінити: Застосувати Part-of-Speech (POS) фільтрацію перед векторизацією: видалити всі займенники та найчастіші іменники ("man", "woman", "person") і залишити тільки дієслова (дії) та специфічні іменники (об'єкти).

2. Проблемна тема: LDA (k=10), Тема 7 (Mixed Topic)
* Яка це тема: Змішана тема. Топ-слова вказують на заклади харчування ("food", "eating", "restaurant"), але топ-документи видають "чоловіка з пакетом покупок" та "жінок з табличкою YMCA". 
* Чому вийшла поганою: LDA групує тексти за спільною появою слів (co-occurrence). Ймовірно, слова "holding" і "bag" в корпусі часто зустрічаються як у контексті їжі (takeaway food bag), так і в контексті шопінгу, що змусило алгоритм злити ці два різні візуальні сюжети в один.
""")


[АНАЛІЗ "ПОГАНИХ" ТЕМ]

1. Проблемна тема: LSA (k=10), Теми 0, 1, 2, 3 та 4 (Duplicate Topics)
* Яка це тема: Майже половина тем у LSA є абсолютними дублікатами. Топ-слова у всіх цих п'яти темах складаються з міксу "man, woman, street, walking, sitting". 
* Чому вийшла поганою: Алгоритм SVD, на якому базується LSA, намагається знайти ортогональні вектори дисперсії. Оскільки в датасеті SNLI кожне речення починається з "A man..." або "A woman...", ці слова мають гігантську вагу в TF-IDF. LSA не зміг "пробитися" крізь цей лексичний шум і просто розмножив одну тему на п'ять.
* Що змінити: Застосувати Part-of-Speech (POS) фільтрацію перед векторизацією: видалити всі займенники та найчастіші іменники ("man", "woman", "person") і залишити тільки дієслова (дії) та специфічні іменники (об'єкти).

2. Проблемна тема: LDA (k=10), Тема 7 (Mixed Topic)
* Яка це тема: Змішана тема. Топ-слова вказують на заклади харчування ("food", "eating", "restaurant"), але топ-документи видають "чоловіка з пакето

In [26]:
# 11. LSA vs LDA comparison

print("""
[ПОРІВНЯННЯ LSA ТА LDA ДЛЯ КОРПУСУ NLI]

1. Читабельність та якість тем
Модель LDA згенерувала значно кращі, вузькоспрямовані сюжети (футбол, музичні гурти, діти в басейні). LSA зазнала повного краху через специфіку датасету (одноманітні структури речень), згенерувавши 5-6 ідентичних тем про "людей на вулиці".

2. Математика проти Логіки (Coherence Score)
Дуже цікавий інсайт: метрика Coherence віддала перевагу моделі LSA (0.3729 при k=5) порівняно з LDA (0.3324 при k=5). Це сталося тому, що слова "man, woman, street" статистично зустрічаються разом майже в кожному реченні корпусу, і математично вони "високоузгоджені". Проте для людини такі теми абсолютно неінформативні. Це доводить, що Coherence не є панацеєю, і ручна інтерпретація завжди має бути фінальним критерієм якості.

3. Висновок
Для аналізу зображень (візуальних сюжетів) ймовірнісна модель LDA (CountVectorizer) набагато краще справляється зі знаходженням прихованих сценаріїв (дій та об'єктів), ігноруючи статистичне домінування загальних слів, об яке "спіткнулася" лінійна модель LSA (TF-IDF).
""")


[ПОРІВНЯННЯ LSA ТА LDA ДЛЯ КОРПУСУ NLI]

1. Читабельність та якість тем
Модель LDA згенерувала значно кращі, вузькоспрямовані сюжети (футбол, музичні гурти, діти в басейні). LSA зазнала повного краху через специфіку датасету (одноманітні структури речень), згенерувавши 5-6 ідентичних тем про "людей на вулиці".

2. Математика проти Логіки (Coherence Score)
Дуже цікавий інсайт: метрика Coherence віддала перевагу моделі LSA (0.3729 при k=5) порівняно з LDA (0.3324 при k=5). Це сталося тому, що слова "man, woman, street" статистично зустрічаються разом майже в кожному реченні корпусу, і математично вони "високоузгоджені". Проте для людини такі теми абсолютно неінформативні. Це доводить, що Coherence не є панацеєю, і ручна інтерпретація завжди має бути фінальним критерієм якості.

3. Висновок
Для аналізу зображень (візуальних сюжетів) ймовірнісна модель LDA (CountVectorizer) набагато краще справляється зі знаходженням прихованих сценаріїв (дій та об'єктів), ігноруючи статистичне домінуванн

In [27]:
# 12. Generate docs/audit_summary_lab8.md


os.makedirs(os.path.join(os.path.dirname(data_dir), 'docs'), exist_ok=True)
summary_path = os.path.join(os.path.dirname(data_dir), 'docs', 'audit_summary_lab8.md')

audit_summary_text = """# Audit Summary Lab 8: Topic Modeling

1. **Розмір корпусу після фільтрації:**
Залишилося документів: 153,607 (всі занадто короткі тексти відкинуто).

2. **Які моделі протестовано:**
* LSA (TF-IDF + TruncatedSVD)
* LDA (CountVectorizer + Latent Dirichlet Allocation)

3. **Які k (кількість тем) протестовано:**
Перевірено k = 5, 8, 10 для обох моделей.

4. **2 найкращі теми (на основі LDA, k=10):**
* **Командні види спорту (Футбол):** Модель ідеально згрупувала слова (soccer, ball, field) та документи з описами гравців на полі.
* **Дитяча активність / Басейни:** Згруповано слова (boy, jumping, blue) та документи про стрибки у воду і дитячий бейсбол.

5. **Найгірша тема (Погана тема):**
* **LSA (k=10), Теми 0-4:** 5 ідентичних тем-дублікатів, що складаються зі слів "man, woman, walking, street". 

6. **Що зіпсувало погану тему:**
Специфіка датасету SNLI. Оскільки всі тексти — це описи фото, вони масово використовують шаблонні конструкції ("A man is..."). LSA через механізм TF-IDF віддала цим словам максимальну вагу і не змогла розгледіти за ними реальні унікальні об'єкти (гітари, м'ячі, собак).

7. **Яка модель краща для цього кейсу:**
Абсолютним переможцем стала модель **LDA**. Незважаючи на нижчий формальний показник Coherence, LDA здатна розпізнавати конкретні візуальні сценарії (одяг, їжа, спорт), тоді як LSA просто групує базові займенники та іменники.

8. **Що робити далі:**
Для покращення результатів у майбутньому необхідно змінити Preprocessing: додати **Part-of-Speech (POS) фільтрацію**. Якщо перед побудовою Topic Model видалити всі загальні іменники (man, woman, person, people) і залишити лише дієслова (дії) та специфічні іменники (об'єкти), якість тем виросте в рази.
"""

with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(audit_summary_text)

print(f"Файл {summary_path} успішно згенеровано.")

Файл ..\docs\audit_summary_lab8.md успішно згенеровано.
